In [1]:
# 1. Import libraries
import pandas as pd, numpy as np

In [2]:
# 2. Load and inspect
df = pd.read_csv('c:\\Users\\babyk\\OneDrive\\Desktop\\Project 1\\data.csv', encoding='latin1')
print(df.shape); df.info()
print(df.isna().sum())
print(df.describe())

(541909, 8)
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64
            Quantity      UnitPrice     CustomerID
count  541909.000000  541909.000000  406829.000000
mean        9.552250       4.611114   15287.690570
std       218.081158      96.759853    1713.600303
min    

In [3]:
# 3. Fixing data types
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='%m/%d/%Y %H:%M')
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df['StockCode'] = df['StockCode'].astype(str).str.upper().str.strip()
df['CustomerID'] = df['CustomerID'].astype('Int64').astype(str).replace('<NA>', np.nan)

In [4]:
# 4. Remove duplicates
n0 = len(df)
df = df.drop_duplicates()
print('Duplicates removed:', n0 - len(df))

Duplicates removed: 5268


In [5]:
# 5. Remove rows with missing values
desc_map = (df.dropna(subset=['Description']).groupby('StockCode')['Description']
              .agg(lambda s: s.mode().iloc[0]))
df['Description'] = df['Description'].fillna(df['StockCode'].map(desc_map))
df = df.dropna(subset=['Description'])
df['CustomerID'] = df['CustomerID'].fillna('Guest')

In [6]:
# 6. Inconsistencies
df['Description'] = df['Description'].str.upper().str.strip().str.replace(r'\s+', ' ', regex=True)
df['Country'] = df['Country'].replace({'EIRE':'Ireland','RSA':'South Africa','USA':'United States'})
df.loc[df['Country']=='Unspecified','Country'] = 'Unknown'

# keep only real products (codes starting with 5 digits); drops POST, DOT, M, BANK CHARGES, gift_ etc.
df = df[df['StockCode'].str.match(r'^\d{5}')]

In [7]:
# 7. Separate returns from sales
df['IsCancelled'] = df['InvoiceNo'].str.startswith('C')
returns = df[df['IsCancelled']].copy()
sales = df[(~df['IsCancelled']) & (df['Quantity']>0) & (df['UnitPrice']>0)].copy()

In [8]:
# 8. Outliers. Check the extremes
sales.nlargest(10,'Quantity')

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancelled
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446,United Kingdom,False
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346,United Kingdom,False
421632,573008,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4800,2011-10-27 12:26:00,0.21,12901,United Kingdom,False
206121,554868,22197,SMALL POPCORN HOLDER,4300,2011-05-27 10:52:00,0.72,13135,United Kingdom,False
97432,544612,22053,EMPIRE DESIGN ROSETTE,3906,2011-02-22 10:43:00,0.82,18087,United Kingdom,False
270885,560599,18007,ESSENTIAL BALM 3.5G TIN IN ENVELOPE,3186,2011-07-19 17:04:00,0.06,14609,United Kingdom,False
52711,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-01-11 12:55:00,2.10,15749,United Kingdom,False
160546,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-04-18 13:20:00,2.10,15749,United Kingdom,False
433788,573995,16014,SMALL CHINESE STYLE SCISSOR,3000,2011-11-02 11:24:00,0.32,16308,United Kingdom,False
4945,536830,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,2880,2010-12-02 17:38:00,0.18,16754,United Kingdom,False


In [9]:
# 9. Feature engineering
sales['Revenue'] = sales['Quantity'] * sales['UnitPrice']
sales['Date']  = sales['InvoiceDate'].dt.date
sales['Year']  = sales['InvoiceDate'].dt.year
sales['Month'] = sales['InvoiceDate'].dt.month
sales['YearMonth'] = sales['InvoiceDate'].dt.to_period('M').astype(str)
sales['Weekday'] = sales['InvoiceDate'].dt.day_name()
sales['Hour'] = sales['InvoiceDate'].dt.hour

region_map = {
 'United Kingdom':'UK',
 'Germany':'Europe','France':'Europe','Ireland':'Europe','Spain':'Europe','Netherlands':'Europe',
 'Belgium':'Europe','Switzerland':'Europe','Portugal':'Europe','Italy':'Europe','Austria':'Europe',
 'Norway':'Europe','Sweden':'Europe','Denmark':'Europe','Finland':'Europe','Poland':'Europe',
 'Channel Islands':'Europe','Iceland':'Europe','Lithuania':'Europe','Cyprus':'Europe','Malta':'Europe',
 'Greece':'Europe','Czech Republic':'Europe','European Community':'Europe',
 'Australia':'Asia-Pacific','Japan':'Asia-Pacific','Singapore':'Asia-Pacific','Hong Kong':'Asia-Pacific',
 'Israel':'Middle East','Saudi Arabia':'Middle East','United Arab Emirates':'Middle East','Lebanon':'Middle East','Bahrain':'Middle East',
 'United States':'Americas','Canada':'Americas','Brazil':'Americas',
 'South Africa':'Africa'}
sales['Region'] = sales['Country'].map(region_map).fillna('Other')

In [10]:
# 10. Create Category
rules = [
 ('Christmas & Seasonal', ['CHRISTMAS','XMAS','SNOW','ADVENT','EASTER','SANTA']),
 ('Lighting & Candles',   ['LIGHT','CANDLE','LANTERN','T-LIGHT','LAMP']),
 ('Kitchen & Dining',     ['MUG','CUP','PLATE','BOWL','TEAPOT','CAKE','JAR','SPOON','KITCHEN','TEA ','BOTTLE','LUNCH']),
 ('Bags & Storage',       ['BAG','BOX','TIN','BASKET','CASE','TRUNK','STORAGE']),
 ('Home Decor',           ['HEART','FRAME','CLOCK','MIRROR','CUSHION','SIGN','WALL','DOORMAT','VINTAGE','DECORATION']),
 ('Stationery & Gifts',   ['CARD','NOTEBOOK','PAPER','WRAP','PENCIL','STICKER','RIBBON','GIFT','TAG']),
 ('Toys & Kids',          ['TOY','DOLL','PLAYHOUSE','CHILDREN','GAME','PUZZLE','BIRD']),
 ('Garden & Outdoor',     ['GARDEN','PLANT','FLOWER','WATERING','BIRDHOUSE']),
]
def categorize(d):
    for cat, kws in rules:
        if any(k in d for k in kws): return cat
    return 'Other'
sales['Category'] = sales['Description'].apply(categorize)
print(sales['Category'].value_counts(normalize=True))

Category
Other                   0.210090
Home Decor              0.206414
Kitchen & Dining        0.191593
Bags & Storage          0.155890
Lighting & Candles      0.067416
Stationery & Gifts      0.065755
Christmas & Seasonal    0.055789
Toys & Kids             0.026358
Garden & Outdoor        0.020696
Name: proportion, dtype: float64


In [11]:

# 11. Confirm how much of the last month you actually have
print('Data ends:', sales['InvoiceDate'].max())
print(sales.groupby('YearMonth')['Revenue'].sum().tail(4))

Data ends: 2011-12-09 12:50:00
YearMonth
2011-09    1028313.80
2011-10    1103310.96
2011-11    1452112.69
2011-12     614487.63
Name: Revenue, dtype: float64


In [12]:
# 12. Flag the last (incomplete) month automatically
last_month = sales['YearMonth'].max()          # '2011-12'
sales['IsPartialMonth'] = sales['YearMonth'] == last_month

In [13]:
# 13. How many days of that month are covered?
days_covered = sales.loc[sales['IsPartialMonth'], 'InvoiceDate'].dt.day.max()
print(f'{last_month} covers only {days_covered} days')

2011-12 covers only 9 days


In [14]:
# 14. Customer table (for segmentation)
cust = sales[sales.CustomerID!='Guest']
snap = sales.InvoiceDate.max() + pd.Timedelta(days=1)
customers = cust.groupby('CustomerID').agg(
    TotalSpend=('Revenue','sum'),
    Orders=('InvoiceNo','nunique'),
    Items=('Quantity','sum'),
    LastPurchase=('InvoiceDate','max'),
    FirstPurchase=('InvoiceDate','min')).reset_index()
customers['RecencyDays'] = (snap - customers['LastPurchase']).dt.days
customers['AvgOrderValue'] = customers['TotalSpend']/customers['Orders']
customers['SpendSegment'] = pd.qcut(customers['TotalSpend'], 4, labels=['Low','Medium','High','VIP'])

In [15]:
#15. Export
sales.to_csv('c:\\Users\\babyk\\OneDrive\\Desktop\\Project 1\\sales_clean.csv', index=False)
returns.to_csv('c:\\Users\\babyk\\OneDrive\\Desktop\\Project 1\\returns.csv', index=False)
customers.to_csv('c:\\Users\\babyk\\OneDrive\\Desktop\\Project 1\\customers.csv', index=False)